<a href="https://colab.research.google.com/github/qahtanaa/OnSubGroupFairness/blob/main/ComparisonwOthers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [753]:
%reset -f

In [754]:
import numpy as np
import pandas as pd
from aif360.metrics import utils
from scipy.sparse import issparse
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedShuffleSplit
import random
from sklearn.neighbors import NearestNeighbors
from sympy import Symbol
from sympy.solvers import solve
from aif360.datasets import BinaryLabelDataset
from aif360.metrics import BinaryLabelDatasetMetric, ClassificationMetric
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from aif360.algorithms.preprocessing import *
from aif360.algorithms.preprocessing.optim_preproc_helpers import distortion_functions, opt_tools
from aif360.algorithms.inprocessing import *
from aif360.algorithms.postprocessing import *
import math
import itertools
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
import time

import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import LSTM
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error



---

---


---



DATASET

In [755]:
def preprocess_dataset(dataset_path, dataset_type, model):
    if dataset_type == 'German':
        df = pd.read_csv(dataset_path)
        df['age'] = df['age'].apply(lambda age: 1 if age >= 25 else 0)
        df['personal_status'] = df['personal_status'].apply(lambda sex: 1 if sex == 'male' else 0)
        #print("German dataset:")
        #print(df.head())
        sensitive_attributes = ['personal_status','age']
        label = 'credit'
        privileged = [1, 1]
        unprivileged = [0, 0]
        favorable_label = 1
        unfavorable_label = 2
        groups = [
                  {'name': 'Male Adult', 'attributes': {'personal_status': 1, 'age': 1}},
                  {'name': 'Female Adult', 'attributes': {'personal_status': 0, 'age': 1}},
                  {'name': 'Male Young', 'attributes': {'personal_status': 1, 'age': 0}},
                  {'name': 'Female Young', 'attributes': {'personal_status': 0, 'age': 0}}
              ]
        model = model

    elif dataset_type == 'COMPAS':
        df = pd.read_csv(dataset_path)
        selected_columns = ['sex', 'age_cat', 'race', 'juv_fel_count', 'juv_misd_count',
                            'juv_other_count', 'priors_count', 'c_charge_degree',
                            'c_charge_desc', 'two_year_recid']
        df = df[selected_columns]
        df = df[(df['race'] == 'Caucasian') | (df['race'] == 'African-American')].reset_index(drop=True)
        #print("COMPAS dataset:")
        #print(df.head())
        sensitive_attributes = ['race','sex']
        label = 'two_year_recid'
        privileged = ['Caucasian', 'Female']
        unprivileged = ['African-American', 'Male']
        favorable_label = 0
        unfavorable_label = 1
        groups = [
                  {'name': 'Caucasian Female', 'attributes': {'race': 1, 'sex': 1}},
                  {'name': 'Black Female', 'attributes': {'race': 0, 'sex': 1}},
                  {'name': 'Causasian Male', 'attributes': {'race': 1, 'sex': 0}},
                  {'name': 'Black Male', 'attributes': {'race': 0, 'sex': 0}}
              ]
        model = model

    elif dataset_type == 'Adult':
        df = pd.read_csv(dataset_path, delimiter=';')
        df['income'] = df['income'].str.strip().replace({'>50K.': '>50K', '<=50K.': '<=50K'})
        df = df.map(lambda x: x.strip() if isinstance(x, str) else x)
        df.replace('?', np.nan, inplace=True)
        df = df.drop(columns=['fnlwgt', 'education-num'])
        df = df[(df['race'] == 'White') | (df['race'] == 'Black')].reset_index(drop=True)
        #print("Adult dataset:")
        #print(df.head())
        sensitive_attributes = ['race','sex']
        label = 'income'
        privileged = ['White', 'Male']
        unprivileged = ['Black', 'Female']
        favorable_label = '>50K'
        unfavorable_label = '<=50K'
        groups = [
                  {'name': 'White Male', 'attributes': {'race': 1, 'sex': 1}},
                  {'name': 'Black Male', 'attributes': {'race': 0, 'sex': 1}},
                  {'name': 'White Female', 'attributes': {'race': 1, 'sex': 0}},
                  {'name': 'Black Female', 'attributes': {'race': 0, 'sex': 0}}
              ]
        model = model
    elif dataset_type == 'Hospital':
        df = pd.read_csv(dataset_path, delimiter=',')
        #print("Hospital dataset:")
        #print(df.head())
        sensitive_attributes = ['gender', 'race']
        label = 'disposition'
        privileged = ['Male', 1]
        unprivileged = ['Female', 0]
        favorable_label = 'Admit'
        unfavorable_label = 'Discharge'
        groups = [
                  {'name': 'White Male', 'attributes': {'race': 1, 'gender': 1}},
                  {'name': 'Others Male', 'attributes': {'race': 0, 'gender': 1}},
                  {'name': 'White Female', 'attributes': {'race': 1, 'gender': 0}},
                  {'name': 'Others Female', 'attributes': {'race': 0, 'gender': 0}}
              ]
        model = model

    return df, sensitive_attributes, label, privileged, unprivileged, \
    		favorable_label, unfavorable_label, groups, model

In [756]:
##################################################################################
# df, sensitive_attributes, label, privileged, unprivileged, favorable_label, unfavorable_label, groups, model = preprocess_dataset('data/raw_german_dataset.csv', 'German', 'Logistic Regression')
# = preprocess_dataset('data/raw_german_dataset.csv', 'German')
# = preprocess_dataset('data/raw_compas_dataset.csv', 'COMPAS')
# = preprocess_dataset('data/raw_adult_dataset.csv', 'Adult')
# = preprocess_dataset('data/raw_hospital_dataset.csv', 'Hospital')

#model == 'Logistic Regression'
#model == 'Random Forest'
#model == 'Gradient Boosting'
#model == 'Deep Neural Network'



---

---

---

DATA PREPARATION

In [757]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

class DataPreparation():
    """
    ........
    """
    def __init__(self, df, sensitive, label, priv, unpriv, fav, unfav, categorical=[]):
        """
        Construct all necessary attributes for the data preparation.

        df : (pandas DataFrame) containing the data
        sensitive : (list(str)) specifying the column names of all sensitive features
        label : (str) specifying the label column
        priv : (list(dicts)) representation of the privileged groups
        unpriv : (list(dicts)) representation of the unprivileged groups
        fav : (str/int/..) value representing the favorable label
        unfav : (str/int/..) value representing the unfavorable label
        categorical : (list(str)) (optional) specifying column names of categorical features
        """
        self.df = df
        self.sensitive = sensitive
        self.label = label
        self.priv = priv
        self.unpriv = unpriv
        self.fav = fav
        self.unfav = unfav
        self.categorical = categorical

    def detect_missing_values(self):
        """
        Detect rows with missing values and remove them from the DataFrame.
        """
        initial_rows = len(self.df)
        self.df = self.df.dropna()
        removed_rows = initial_rows - len(self.df)

        if removed_rows > 0:
            print(f"Detected {removed_rows} rows with missing values. Removed them.")
        else:
            print("No missing values detected.")  # pass

    def binary_label(self):
        """
        Ensure the decision label and sensitive attributes are encoded as binary, where:
        - Favorable label and privileged groups are encoded as 1.
        - Unfavorable label and unprivileged groups are encoded as 0.
        """
        if len(self.priv) != 2 or len(self.unpriv) != 2:
            raise ValueError("Both 'priv' and 'unpriv' must contain exactly two values.")

        number_label_values = self.df[self.label].nunique()
        if number_label_values == 2:
            print(f"The '{self.label}' column has only two unique values.")
            self.df[self.label] = (self.df[self.label].map({self.unfav: 0, self.fav: 1}).astype(int))
        else:
            print(f"The '{self.label}' column does not have exactly two unique values, as it should.")

        # Create mappings for each sensitive attribute
        race_mapping = {self.priv[0]: 1, self.unpriv[0]: 0}
        sex_mapping = {self.priv[1]: 1, self.unpriv[1]: 0}

        # Apply the mappings to the respective columns
        self.df[self.sensitive[0]] = (
            self.df[self.sensitive[0]]
            .replace(race_mapping)
        )

        self.df[self.sensitive[1]] = (
            self.df[self.sensitive[1]]
            .replace(sex_mapping)
        )

        self.df[self.sensitive[0]] = self.df[self.sensitive[0]].astype(int)
        self.df[self.sensitive[1]] = self.df[self.sensitive[1]].astype(int)

    def find_categorical_attributes(self):
        """
        Identify categorical attributes and encode.
        """
        self.attribute_types = {}

        for column in self.df.columns:
            if column == 'Group':
                continue  # Skip the 'Group' column
            elif column in self.categorical:
                self.attribute_types[column] = 'Categorical'
            elif self.df[column].nunique() == 2:
                self.attribute_types[column] = 'Categorical'
            else:
                num_float = 0
                num_text = 0
                thresh = 0.99
                num_att_in_column = len(self.df[column])

                for value in self.df[column]:
                    try:
                        float(value)
                        num_float += 1
                    except ValueError:
                        num_text += 1

                if num_float / num_att_in_column > thresh:
                    self.attribute_types[column] = 'Numerical'
                else:
                    self.attribute_types[column] = 'Categorical'
        # Boolean
        self.cat_features = []
        for attr in self.attribute_types:
            self.cat_features.append(self.attribute_types[attr] == 'Categorical')

        encoder_dict = dict()
        self.columns_categorical = self.df.columns[self.cat_features]

        for column in self.columns_categorical:
            le = LabelEncoder()
            encoded = le.fit_transform(self.df[column].astype(str).values)
            self.df[column] = pd.Series(encoded, index=self.df.index, dtype="int64")
            mapping = dict(zip(le.classes_, range(len(le.classes_))))
            encoder_dict[column] = mapping
        self.numerical_features = [not feature for feature in self.cat_features]
        self.columns_numerical = self.df.columns[self.numerical_features]

        for column in self.columns_numerical:
            self.df.loc[:, column] = self.df[column].astype(float)

        return self.attribute_types, self.cat_features, self.numerical_features

    def create_group_column(self):
        """
        Create a 'Group' column in the DataFrame based on protected attributes, privileged/unprivileged conditions, and label.
        """
        group_combinations = pd.MultiIndex.from_product([self.df[sensitive].unique() for sensitive in self.sensitive] + [self.df[self.label].unique()], names=self.sensitive + [self.label])
        #print(list(enumerate(group_combinations)))
        # Create a mapping between group combinations and their corresponding numbers
        group_mapping = {group: idx for idx, group in enumerate(group_combinations)}
        reverse_group_mapping = {idx: group for group, idx in group_mapping.items()}
        self.df['Group'] = pd.MultiIndex.from_frame(self.df[self.sensitive + [self.label]]).map(group_mapping)

        return reverse_group_mapping

    def train_test_split(self):
        X = self.df.loc[:, self.df.columns != self.label]
        y = self.df[self.label]
        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(X, y, test_size=0.3, shuffle=True, stratify=self.df['Group'])  # , random_state=42

        return self.X_train, self.y_train, self.X_test, self.y_test

    def standardization_numerical(self):

        numerical_cols = [
            col for col in self.X_train.columns
            if col in self.columns_numerical
        ]

        train_dataset_numerical = self.X_train[numerical_cols]
        test_dataset_numerical = self.X_test[numerical_cols]
        scaler = StandardScaler()
        scaler.fit(train_dataset_numerical)
        self.X_train[numerical_cols] = scaler.transform(train_dataset_numerical)
        self.X_test[numerical_cols] = scaler.transform(test_dataset_numerical)
        self.X_train = pd.concat(
            [self.X_train.reset_index(drop=True),
             self.y_train.reset_index(drop=True)],
            axis=1
        )

        self.X_test = pd.concat(
            [self.X_test.reset_index(drop=True),
             self.y_test.reset_index(drop=True)],
            axis=1
        )

        return self.X_train, self.X_test

    def prepare(self):
        """
        Perform all preprocessing steps.
        """
        self.detect_missing_values()
        self.binary_label()
        self.find_categorical_attributes()
        self.create_group_column()
        self.train_test_split()
        self.standardization_numerical()
        return self


In [758]:
def compute_IR(train_df, s_attr, label):
    s0, s1 = s_attr

    # groupby compatto (1 sola passata)
    counts = (
        train_df
        .groupby([s0, s1, label], observed=True)
        .size()
        .unstack(fill_value=0)
    )

    # funzione di supporto
    def ir(a, b):
        return a / b if b != 0 else float("inf")

    imbalance_ratios = {}

    for g0 in [0, 1]:
        for g1 in [0, 1]:
            ones = counts.loc[(g0, g1), 1] if (g0, g1) in counts.index else 0
            zeros = counts.loc[(g0, g1), 0] if (g0, g1) in counts.index else 0
            imbalance_ratios[(g0, g1)] = ir(ones, zeros)

    return imbalance_ratios

In [759]:
# ##################################################################################
# #use the DataPreparation class to preprocess the dataframe
# data_prep = DataPreparation(df, sensitive_attributes, label, privileged, unprivileged, favorable_label, unfavorable_label)
# data_prep.prepare()
# data_prep.df = data_prep.df.reset_index(drop=True)
# X_train, X_test = data_prep.X_train, data_prep.X_test
# attribute_types = data_prep.attribute_types
# cat_features = data_prep.cat_features
# numerical_features = data_prep.numerical_features
# reverse_group_mapping = data_prep.create_group_column()
# #theoretical_num_groups = len(reverse_group_mapping)
# X_train = X_train.reset_index(drop=True)


# imbalance_ratios = compute_IR(X_train, sensitive_attributes, label)
# # test(X_train)
# # print (len(data_prep.X_train))

In [760]:
# s0, s1 = sensitive_attributes

# counts = (
#     X_train
#     .groupby([s0, s1, label], observed=True)
#     .size()
#     .unstack(fill_value=0)
# )

# # gruppo privilegiato (1,1)
# ones = counts.loc[(1, 1), 1] if (1, 1) in counts.index else 0
# zeros = counts.loc[(1, 1), 0] if (1, 1) in counts.index else 0

# total_ratio = ones / zeros if zeros != 0 else float("inf")

# print(f"Ratio of most privileged class: {total_ratio}")

In [761]:
# #################################################################################
# subgroup_column_train = X_train.pop('Group')
# subgroup_column_test = X_test.pop('Group')

In [762]:
# X_train.head()



---



---



---

DBSCAN



In [763]:
import math

In [764]:
from gower import gower_matrix
from sklearn.cluster import DBSCAN
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors

In [765]:
def stratified_subsample(df, group_col, max_size=10000, random_state=42):
    """
    Estrae un sottocampione stratificato rispetto a group_col
    mantenendo la distribuzione dei gruppi.
    """
    if len(df) <= max_size:
        return df

    # Compute proportions of groups
    group_counts = df[group_col].value_counts(normalize=True)

    sampled_parts = []
    for group, proportion in group_counts.items():
        n_group_samples = max(1, int(round(proportion * max_size)))
        group_df = df[df[group_col] == group]

        sampled_parts.append(
            group_df.sample(
                n=min(n_group_samples, len(group_df)),
                random_state=random_state
            )
        )

    sampled_df = pd.concat(sampled_parts)

    if len(sampled_df) > max_size:
        sampled_df = sampled_df.sample(n=max_size, random_state=random_state)
    elif len(sampled_df) < max_size:
        remaining = max_size - len(sampled_df)
        sampled_df = pd.concat([
            sampled_df,
            df.drop(sampled_df.index).sample(
                n=min(remaining, len(df) - len(sampled_df)),
                random_state=random_state
            )
        ])

    return sampled_df.reset_index(drop=True)

In [766]:
def find_optimal_epsilon(filtered_data, cat_features, min_samples, distance_matrix, eps_step=0.001, eps_min=0.01, eps_max=1.1):

    def cluster_count(eps):
        dbscan = DBSCAN(eps=eps, min_samples=min_samples, metric='precomputed')
        labels = dbscan.fit_predict(distance_matrix)
        unique_labels = np.unique(labels)
        n_clusters = len(unique_labels)  # - (1 if -1 in unique_labels else 0)
        return n_clusters, unique_labels

    # Binary search for optimal epsilon
    while eps_max - eps_min > eps_step:
        eps_mid = (eps_min + eps_max) / 2
        n_clusters_mid, labels_mid = cluster_count(eps_mid)
        #print(eps_mid, n_clusters_mid, labels_mid, 'mids')

        if n_clusters_mid == 1:
            if -1 in labels_mid:
                eps_min = eps_mid  # Only noise points, increase epsilon
            else:
                eps_max = eps_mid  # Only core points, decrease epsilon
        elif n_clusters_mid > 2:
            eps_min = eps_mid  # More than two clusters, increase epsilon
        else:
            eps_max = eps_mid  # Exactly two clusters, continue search to fine-tune
        #print(eps_min, eps_max, 'min, max')
    return eps_max

# Custom SMOTE-DBSCAN function
def custom_smote_dbscan(X_train, cat_features, pu_ix, nu_ix, group_column_train, total_ratio):
    """
    X_train is the training dataset preprocessed, group_column_train is a column containing the group of each
    instance in X_train
    """
    cat_attr_ix = [i for i, value in enumerate(cat_features) if value]

    X2_df = X_train[group_column_train == pu_ix]
    X2 = X2_df.values
    X3_df = X_train[group_column_train == nu_ix]
    X3 = X3_df.values

    PU = len(X2)
    NU = len(X3)

    # Determine the oversampling target based on a given total_ratio
    if (PU / NU) > total_ratio:
        oversampling_target = (PU / total_ratio) - NU
        os_df = X3_df
        os_ix = nu_ix
    elif (PU / NU) == total_ratio:
        print("The ratio of PU to NU is within the acceptable range of total_ratio.")
        return [], 0, pu_ix
    else:
        oversampling_target = (total_ratio * NU) - PU
        os_df = X2_df
        os_ix = pu_ix
    os_df = os_df.reset_index(drop=True)
    
    # === stratified subsampling ===
    MAX_GOWER_SIZE = 5000
    # if len(os_df) > MAX_GOWER_SIZE:
    #     os_df = stratified_subsample(
    #         df=os_df.assign(Group=group_column_train.loc[os_df.index].values),
    #         group_col="Group",
    #         max_size=MAX_GOWER_SIZE
    #     ).drop(columns=["Group"])
    
    # compute min_samples on dataframe  ===Hakim for sensitivity analysis
    
    # min_samples = 5
    # min_samples = 10
    # min_samples = len(os_df.columns)
    # min_samples = 2 * len(os_df.columns)
    min_samples = round(math.log(len(os_df)))
    
    # print ('Columns = ', len(os_df.columns))

    print('min_samples', min_samples)

    distance_matrix = gower_matrix(os_df, cat_features=cat_features)

    # Find the optimal epsilon for os_df
    optimal_eps = find_optimal_epsilon(os_df, cat_features, min_samples, distance_matrix)

    # DBSCAN clustering with the optimal epsilon
    dbscan = DBSCAN(eps=optimal_eps, min_samples=min_samples, metric='precomputed')
    clusters = dbscan.fit_predict(distance_matrix)

    # Get cluster labels
    labels = dbscan.labels_

    # Get core samples
    core_samples_mask = np.zeros_like(labels, dtype=bool)
    core_samples_mask[dbscan.core_sample_indices_] = True

    # Identify core, border, and noise points
    core_points = os_df[core_samples_mask]
    border_points = os_df[~core_samples_mask & (labels != -1)]
    noise_points = os_df[labels == -1]

    if len(border_points) == 0:
        border_points = core_points

    # Initialize synthetic samples list
    synthetic_samples = []

    border_indices = border_points.index.tolist()
    random.shuffle(border_indices)
    current_index = 0

    while len(synthetic_samples) < oversampling_target:
        idx_A = border_indices[current_index % len(border_indices)]
        current_index += 1
        point_A = os_df.loc[idx_A]

        # Ensure point B is not a noise point
        distances_to_A = distance_matrix[idx_A]
        neighbors = np.argsort(distances_to_A)[1:min_samples+1]  # Exclude the point itself
        valid_neighbors = [idx for idx in neighbors if labels[idx] != -1]  # Exclude noise points

        if not valid_neighbors:
            continue  # Skip if no valid neighbors are found

        idx_B = np.random.choice(valid_neighbors)
        point_B = os_df.loc[idx_B]

        synthetic_point = {}
        for i, col in enumerate(os_df.columns):
            if cat_features[i]:
                neighbor_values = os_df.iloc[valid_neighbors][col].tolist()
                synthetic_point[col] = max(set(neighbor_values), key=neighbor_values.count)
            else:
                alpha = np.random.rand()
                synthetic_point[col] = point_A[col] + alpha * (point_B[col] - point_A[col])

        synthetic_samples.append(synthetic_point)

    return pd.DataFrame(synthetic_samples), len(synthetic_samples), os_ix

# Example usage (assuming you have defined X_train, cat_features, etc.):
# synthetic_samples, num_samples, oversampled_index = custom_smote_dbscan(X_train, cat_features, pu_ix, nu_ix, group_column_train, total_ratio)


In [767]:
def oversample_groups(X_train, cat_features, custom_smote, group_column_train, total_ratio, reverse_group_mapping):
    """
    Function to oversample multiple groups automatically based on group labels.

    Parameters:
    - X_train: Preprocessed training dataset.
    - cat_features: List indicating categorical features.
    - custom_smote: Custom SMOTE function to be used.
    - group_column_train: Column containing the group label for each instance.
    - total_ratio: Desired ratio of positive to negative labels.
    - reverse_group_mapping: Mapping of groups to sensitive attributes and labels.

    Returns:
    - synthetic_samples_matrix: Matrix containing all generated synthetic samples.
    - synthetic_samples_group: Array of group labels for the synthetic samples.
    """

    synthetic_samples = []
    synthetic_samples_group = []

    groups = sorted(group_column_train.unique())
    paired_groups = [(groups[i], groups[i+1]) for i in range(0, len(groups), 2)]

    for group1, group2 in paired_groups:
        ########## Determine pu_ix and nu_ix using reverse_group_mapping ##########
        if reverse_group_mapping[group1][2] == 1:
            pu_ix = group1
            nu_ix = group2
        else:
            pu_ix = group2
            nu_ix = group1
        ##########################################################################

        group_df_pu = X_train[group_column_train == pu_ix]
        group_df_nu = X_train[group_column_train == nu_ix]
        positive_count = group_df_pu[group_df_pu[label] == 1].shape[0]
        negative_count = group_df_nu[group_df_nu[label] == 0].shape[0]

        if positive_count == 0 or negative_count == 0:
            continue

        current_ratio = positive_count / negative_count

        if current_ratio == total_ratio:
            continue  # Skip the most privileged group

        synthetic_points, synthetic_count, os_ix = custom_smote(X_train, cat_features, pu_ix, nu_ix, group_column_train, total_ratio=total_ratio)
        pu_column = np.full((len(synthetic_points), 1), os_ix)
        synthetic_samples.append(synthetic_points)
        synthetic_samples_group.append(pu_column)
        #print(f"Oversampling for group pair ({pu_ix}, {nu_ix}): Added {synthetic_count} synthetic samples in {os_ix}.")

    synthetic_samples_matrix = pd.concat(synthetic_samples, ignore_index=True)
    synthetic_samples_group = np.concatenate(synthetic_samples_group)

    return synthetic_samples_matrix, synthetic_samples_group


In [768]:
# synthetic_samples_matrix_dbscan, synthetic_samples_group_dbscan = oversample_groups(X_train, cat_features, custom_smote_dbscan, subgroup_column_train, total_ratio, reverse_group_mapping)

# # Concatenate the original dataset with the synthetic samples
# X_train_resampled_dbscan = pd.concat([X_train, pd.DataFrame(synthetic_samples_matrix_dbscan, columns=X_train.columns)], ignore_index=True)
# #subgroup_column_resampled_tax = pd.concat([X_train[group_column_train], pd.Series(synthetic_samples_group_tax.flatten())], ignore_index=True)


In [769]:
# X_train



---



---



---
CLASSIFICATION


In [770]:
def evaluate_model_performance(X_train, X_test, protected_attributes, label_name, groups, model, weights=None):
    favorable_label = 1.0
    unfavorable_label = 0.0
    X_train[label_name] = X_train[label_name].astype(float)
    X_test[label_name] = X_test[label_name].astype(float)
    # If weights is not provided, create an array of ones with the same length as X_train
    if weights is None:
        weights = np.ones(len(X_train))

    # Create BinaryLabelDatasets
    binary_ds_train = BinaryLabelDataset(df=X_train, label_names=[label_name],
                                         protected_attribute_names=protected_attributes,
                                         favorable_label=favorable_label, unfavorable_label=unfavorable_label)
    binary_ds_test = BinaryLabelDataset(df=X_test, label_names=[label_name],
                                        protected_attribute_names=protected_attributes,
                                        favorable_label=favorable_label, unfavorable_label=unfavorable_label)
    if model == 'Logistic Regression':
        classifier = LogisticRegression(max_iter = 500)
    elif model == 'Random Forest':
        classifier = RandomForestClassifier(n_estimators=100, random_state=42)
    elif model == 'Gradient Boosting':
        classifier = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
    elif model == 'Neural Network':
        classifier = MLPClassifier(solver='lbfgs', alpha=1e-5, hidden_layer_sizes=(5, 2), max_iter=2000, random_state=1)
    else:
        raise ValueError('Choose one classification algorithm between Logistic Regression, Random Forest, Gradient Boosting, and Neural Network')

    if model != 'Deep Neural Network':

        if model == 'Neural Network':
            classifier.fit(
                X_train.drop(columns=[label_name]),
                X_train[label_name]
            )
        elif model == 'LSTM':
            classifier.fit(
                X_train.drop(columns=[label_name]), 
                X_train[label_name], 
                epochs=3, batch_size=64
            )
        else:
            classifier.fit(
                X_train.drop(columns=[label_name]),
                X_train[label_name],
                sample_weight=weights
            )

    predicted_labels = classifier.predict(
        X_test.drop(columns=[label_name])
    )

    X_test_with_predictions = pd.concat([X_test.drop(columns=[label_name]), pd.Series(predicted_labels, name=label_name, index=X_test.index)], axis=1)

    binary_ds_test_pred = BinaryLabelDataset(df=X_test_with_predictions, label_names=[label_name],
                                             protected_attribute_names=protected_attributes,
                                             favorable_label=favorable_label, unfavorable_label=unfavorable_label)

    all_results = {}
    for (group1, group2) in itertools.combinations(groups, 2):
        #print(group1, group2, 'gruppi')
        pair_key = f"{group1['name']} vs {group2['name']}"
        all_results[pair_key] = evaluate(
            binary_ds_test, binary_ds_test_pred,
            [group1['attributes']], [group2['attributes']])
        #print([group1['attributes']], [group2['attributes']])


    return all_results, predicted_labels

In [771]:
def evaluate(test_data, pred, priv_group, unpriv_group):
    cm = ClassificationMetric(test_data, pred,
                              unprivileged_groups=unpriv_group,
                              privileged_groups=priv_group)
    dm = BinaryLabelDatasetMetric(pred,
                                  unprivileged_groups=unpriv_group,
                                  privileged_groups=priv_group)
    #print(cm, dm, "cm e dm")
    measure_scores = {
        'Balanced Accuracy': balanced_accuracy_score(test_data.labels, pred.labels),
        'Accuracy': cm.accuracy(),
        'F1 Score': f1_score(test_data.labels.ravel(), pred.labels.ravel()),  # Ensure labels are flat
        'Disparate Impact Ratio': dm.disparate_impact(),
        #'Demographic Parity Difference': cm.statistical_parity_difference(),
        #'Predictive Parity Difference': cm.positive_predictive_value(privileged=True) - cm.positive_predictive_value(privileged=False),
        'Average Odds Difference': cm.average_odds_difference(),
        'Equal Opportunity Difference': cm.equal_opportunity_difference(),
        #'Equalized Odds Difference': cm.average_abs_odds_difference(),
        'Consistency': dm.consistency(),
        #'TPR Difference': cm.true_positive_rate_difference(),
        #'FPR Difference': cm.false_positive_rate_difference(),
        #'TNR Difference': cm.true_negative_rate(privileged=True) - cm.true_negative_rate(privileged=False),
        #'FNR Difference': cm.false_negative_rate_difference(),
    }

    return measure_scores

In [772]:
def compute_metrics(df, actual_labels, predicted_labels):
    """Compute fairness and performance metrics."""
    cm = confusion_matrix(actual_labels, predicted_labels)
    TN, FP, FN, TP = cm.ravel()
    metrics = {
        'Accuracy': accuracy_score(actual_labels, predicted_labels),
        'Precision': precision_score(actual_labels, predicted_labels),
        'Recall': recall_score(actual_labels, predicted_labels),
        'F1 Score': f1_score(actual_labels, predicted_labels),
        'TPR': TP / (TP + FN),
        'FPR': FP / (FP + TN),
        'TNR': TN / (TN + FP),
        'FNR': FN / (FN + TP),
        'TP': TP,
        'FP': FP,
        'TN': TN,
        'FN': FN
    }

    return metrics

In [773]:
# ##################################################################################
# results_orig, pred_labels_orig = evaluate_model_performance(X_train, X_test, sensitive_attributes, label,
#                                                             groups, model=model)
# #model = 'Random Forest'
# #model = 'Gradient Boosting'

# # Initialize a list to hold DataFrames
# data_frames = []

# # Populate the list with DataFrames, each having a unique row index
# for key, values in results_orig.items():
#     df_part = pd.DataFrame([values], index=[key])
#     data_frames.append(df_part)

# # Concatenate all DataFrames into a single DataFrame
# results_orig_df = pd.concat(data_frames)
# results_orig_df.index.name = 'Comparison'

# # Print the results DataFrame
# results_orig_df

In [774]:
# ##################################################################################
# results_dbscan, pred_labels_dbscan = evaluate_model_performance(X_train_resampled_dbscan, X_test, sensitive_attributes, label,
#                                                             groups, model=model)

# # Initialize a list to hold DataFrames
# data_frames = []

# # Populate the list with DataFrames, each having a unique row index
# for key, values in results_dbscan.items():
#     df_part = pd.DataFrame([values], index=[key])
#     data_frames.append(df_part)

# # Concatenate all DataFrames into a single DataFrame
# results_dbscan_df = pd.concat(data_frames)
# results_dbscan_df.index.name = 'Comparison'




---



---



---

OTHER MITIGATION ALGORITHMS - FAIR-SMOTE, REWEIGHING, GERRYFAIR, REMEDY



---



---



---

COMPARE


In [775]:
class ModelEvaluator:
    def __init__(self, df, protected_attributes, label_name, privileged, unprivileged, fav, 
                 unfav, groups, model, num_iterations=10, oversampling_methods=None):
        self.df = df
        self.model = model
        self.protected_attributes = protected_attributes
        self.label_name = label_name
        self.privileged = privileged
        self.unprivileged = unprivileged
        self.fav = fav
        self.unfav = unfav
        self.groups = groups
        self.num_iterations = num_iterations
        self.oversampling_methods = oversampling_methods if oversampling_methods is not None else ['none']
        


    def evaluate_model_performance_mean(self):

        results_dict = {method: [] for method in self.oversampling_methods}
        time_dict = {method: [] for method in self.oversampling_methods}

        for _ in range(self.num_iterations):

            # ================= DATA PREPARATION =================

            data_prep = DataPreparation(
                self.df,
                self.protected_attributes,
                self.label_name,
                self.privileged,
                self.unprivileged,
                self.fav,
                self.unfav
            )

            data_prep.prepare()
            data_prep.df = data_prep.df.reset_index(drop=True)

            X_train, X_test = data_prep.X_train, data_prep.X_test
            X_train = X_train.reset_index(drop=True)

            cat_features = data_prep.cat_features
            reverse_group_mapping = data_prep.create_group_column()

            subgroup_column_train = X_train['Group']
            subgroup_column_test = X_test['Group']

            X_train = X_train.drop(columns=['Group'])
            X_test = X_test.drop(columns=['Group'])

            # ================= RATIO CALCULATION =================

            num_privileged_ones = X_train[
                (X_train[self.protected_attributes[0]] == 1) &
                (X_train[self.protected_attributes[1]] == 1) &
                (X_train[self.label_name] == 1)
            ].shape[0]

            num_privileged_zeros = X_train[
                (X_train[self.protected_attributes[0]] == 1) &
                (X_train[self.protected_attributes[1]] == 1) &
                (X_train[self.label_name] == 0)
            ].shape[0]

            total_ratio = (
                num_privileged_ones / num_privileged_zeros
                if num_privileged_zeros != 0 else float('inf')
            )

            # ================= METHODS LOOP =================

            for method in self.oversampling_methods:

                start_time = time.time()
                X_train_method = X_train.copy()

                # ---------- NONE ----------
                if method == 'none':

                    results, _ = evaluate_model_performance(
                        X_train_method, X_test,
                        self.protected_attributes,
                        self.label_name,
                        self.groups,
                        model=self.model
                    )

                # ---------- CUSTOM SMOTE DBSCAN ----------
                elif method == 'custom_smote_dbscan':

                    synthetic_samples_matrix, synthetic_groups = oversample_groups(
                        X_train_method,
                        cat_features,
                        custom_smote_dbscan,
                        subgroup_column_train,
                        total_ratio,
                        reverse_group_mapping
                    )

                    if len(synthetic_samples_matrix) > 0:

                        X_train_resampled = pd.concat(
                            [
                                X_train_method,
                                pd.DataFrame(
                                    synthetic_samples_matrix,
                                    columns=X_train.columns
                                )
                            ],
                            ignore_index=True
                        )

                    else:
                        X_train_resampled = X_train_method

                    results, _ = evaluate_model_performance(
                        X_train_resampled, X_test,
                        self.protected_attributes,
                        self.label_name,
                        self.groups,
                        model=self.model
                    )
    
                # ---------- FAIR-SMOTE ----------
                elif method == 'Fair-SMOTE':
    
                    X_train_resampled = oversample_fair_smote(
                        X_train_method,
                        self.protected_attributes,
                        self.label_name
                    )
    
                    results, _ = evaluate_model_performance(
                        X_train_resampled, X_test,
                        self.protected_attributes,
                        self.label_name,
                        self.groups,
                        model=self.model
                    )
    
                # ---------- REWEIGHING ----------
                elif method == 'Reweighing':
    
                    custom_data = CustomDataset(
                        X_train_method,
                        self.protected_attributes,
                        self.label_name
                    )
    
                    privileged_groups = [{
                        self.protected_attributes[0]: 1,
                        self.protected_attributes[1]: 1
                    }]
    
                    unprivileged_groups = [{
                        self.protected_attributes[0]: 0,
                        self.protected_attributes[1]: 0
                    }]
    
                    RW = Reweighing(
                        unprivileged_groups=unprivileged_groups,
                        privileged_groups=privileged_groups
                    )
    
                    RW.fit(custom_data)
    
                    results, _ = evaluate_model_performance(
                        X_train_method, X_test,
                        self.protected_attributes,
                        self.label_name,
                        self.groups,
                        model=self.model,
                        weights=custom_data.instance_weights
                    )
    
                # ---------- REMEDY ----------
                elif method == 'Remedy':

                    columns_all = X_train_method.drop(columns=[self.label_name])
                    label_y = self.label_name
                    columns_protected = self.protected_attributes

                    temp2, names = get_temp(X_train_method, columns_protected, label_y)

                    unfair_group, unfair_names, skew_candidates, unfair_dict = get_unfair_group(columns_protected, [])

                    all_names = candidate_groups(
                        skew_candidates,
                        unfair_dict,
                        columns_protected,
                        unfair_names
                    )

                    names_values = name_val_dict(X_train_method, names)

                    all_names_lst = list(all_names.keys())[1:]
                    all_names_lst.reverse()

                    filter_count = 30
                    new_train_data = copy.deepcopy(X_train_method)

                    for a in all_names_lst:

                        temp2, names = get_temp(new_train_data, all_names[a], label_y)
                        temp, temp_g = get_temp_g(new_train_data, names, label_y)

                        temp_g = temp_g[temp_g['cnt'] > filter_count]

                        lst_of_counts = compute_lst_of_counts(temp, names, label_y)

                        need_pos, need_neg = compute_problematic_opt(
                            temp2,
                            temp_g,
                            names,
                            label_y,
                            lst_of_counts
                        )

                        new_train_data['skewed'] = 0
                        new_train_data["diff"] = 0

                        new_train_data = naive_duplicate(
                            new_train_data,
                            temp2,
                            names,
                            need_pos,
                            need_neg,
                            label_y
                        )

                    X_train_resampled_remedy = new_train_data.drop(columns=['skewed', 'diff'])

                    results, _ = evaluate_model_performance(
                        X_train_resampled_remedy,
                        X_test,
                        self.protected_attributes,
                        self.label_name,
                        self.groups,
                        model=self.model
                    )

                else:
                    continue


                # ================= SAVE RESULTS =================

                end_time = time.time()
                elapsed_time = end_time - start_time

                data_frames = []

                for key, values in results.items():

                    df_part = pd.DataFrame([values], index=[key])
                    data_frames.append(df_part)

                results_df = pd.concat(data_frames)
                results_df.index.name = 'Comparison'

                results_dict[method].append(results_df)
                time_dict[method].append(elapsed_time)


        # ================= AVERAGE OVER ITERATIONS =================

        combined_results = {}
    
        for method in self.oversampling_methods:
    
            df_mean = pd.concat(results_dict[method]).groupby(level=0).mean()
            avg_time = np.mean(time_dict[method])
    
            df_mean["Running Time (s)"] = avg_time
            combined_results[method] = df_mean
    
        return combined_results

In [776]:
# evaluators = {
#     'ModelEvaluator': ModelEvaluator(df, sensitive_attributes, label, privileged, 
#                                      unprivileged, favorable_label, unfavorable_label, 
#                                      groups, model, num_iterations = 1, 
#                                      oversampling_methods=['none', 'custom_smote_dbscan', 'Fair-SMOTE', 'Reweighing', 'Remedy'])
# }

# results = {}

# # Evaluate all models and store results
# for key, evaluator in evaluators.items():
#     results[key] = evaluator.evaluate_model_performance_mean()

# # Display the results
# #for method, result in results['ModelEvaluator'].items():
#     #print(f"Results for {method} method:")
#     #print(result)

In [777]:

# comparison_key = 'Male Adult vs Female Young'
# #comparison_key = 'Caucasian Female vs Black Male'
# #comparison_key = 'White Male vs Black Female'

# # Create an empty list to store the extracted data
# data = []

# # Iterate over the results dictionary and extract the relevant data
# for method, result_dict in results['ModelEvaluator'].items():
#     if comparison_key in result_dict.index:
#         metrics = result_dict.loc[comparison_key]
#         row = {
#             'Classifier': model,
#             'Technique': method,
#             'DI Ratio': metrics['Disparate Impact Ratio'],
#             'AEO Diff.': metrics['Average Odds Difference'],
#             'Equal Opportunity Difference': metrics['Equal Opportunity Difference'],
#             'Consistency': metrics['Consistency'][0],
#             'Acc.': metrics['Accuracy'],
#             'Bal. Acc.': metrics['Balanced Accuracy'],
#             'F1 Score': metrics['F1 Score']
#         }
#         data.append(row)

# # Convert the extracted data into a DataFrame
# df_results = pd.DataFrame(data)

# # Print the DataFrame as a formatted table
# #print(df_results.to_string(index=False))


# ALL 


In [778]:
# ======================================================
# FULL EXPERIMENT: ALL DATASETS × ALL MODELS × ALL METHODS
# WITH RUNNING TIME
# ======================================================

import pandas as pd
import numpy as np

datasets = {
    # "German": "data/raw_german_dataset.csv",
    # "COMPAS": "data/raw_compas_dataset.csv",
    "Adult": "data/raw_adult_dataset.csv",
    # "Hospital": "data/raw_hospital_dataset2.csv"
}

models = [
    "Logistic Regression",
    # "Random Forest",
    # "Gradient Boosting",
    # "Neural Network",
    #"LSTM"
]

methods_list = [
    # 'none',
    # 'Fair-SMOTE',
    # 'Reweighing',
    'custom_smote_dbscan',
    # 'Remedy'
]

# MinPts = [5, 'LogPos', 'm', '2m']

num_iterations = 10



all_results = []

for dataset_name, path in datasets.items():

    print(f"\n================ DATASET: {dataset_name} ================\n")

    for model in models:

        print(f"\n----- MODEL: {model} -----\n")

        # ================= PREPROCESS =================
        df, sensitive_attributes, label, privileged, unprivileged, \
        favorable_label, unfavorable_label, groups, _ = preprocess_dataset(
            path, dataset_name, model
        )

        evaluator = ModelEvaluator(
            df,
            sensitive_attributes,
            label,
            privileged,
            unprivileged,
            favorable_label,
            unfavorable_label,
            groups,
            model=model,
            num_iterations=num_iterations,
            oversampling_methods=methods_list
        )

        results = evaluator.evaluate_model_performance_mean()

        # ================= FLATTEN RESULTS =================
        for method, result_df in results.items():

            for comparison in result_df.index:

                metrics = result_df.loc[comparison]

                row = {
                    "Dataset": dataset_name,
                    "Model": model,
                    "Method": method,
                    "Comparison": comparison,
                    "Running Time (s)": metrics["Running Time (s)"],
                    "DI Ratio": metrics["Disparate Impact Ratio"],
                    "AEO Diff": metrics["Average Odds Difference"],
                    "EOD": metrics["Equal Opportunity Difference"],
                    "Consistency": metrics["Consistency"][0],
                    "Accuracy": metrics["Accuracy"],
                    "Balanced Accuracy": metrics["Balanced Accuracy"],
                    "F1 Score": metrics["F1 Score"]
                }

                all_results.append(row)


# ======================================================
# FINAL DATAFRAME
# ======================================================
summary_df = (
    pd.DataFrame(all_results)
    .groupby(["Dataset", "Model", "Method"], as_index=False)
    .mean(numeric_only=True)
    .sort_values(by=["Dataset", "Model", "Method"])
    .reset_index(drop=True)
)

#print("\n\n================ FINAL SUMMARY RESULTS ================\n")
#display(summary_df)

# Save to CSV (optional)
# summary_df.to_csv("final_summary_4x4x5.csv", index=False)


================ DATASET: Adult ================


----- MODEL: Logistic Regression -----

Detected 3316 rows with missing values. Removed them.
The 'income' column has only two unique values.


/var/folders/7c/thwjjxtx3_n8jl9xncdtz7rc0000gq/T/ipykernel_77967/330282448.py:56: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.df[self.label] = (self.df[self.label].map({self.unfav: 0, self.fav: 1}).astype(int))
/var/folders/7c/thwjjxtx3_n8jl9xncdtz7rc0000gq/T/ipykernel_77967/330282448.py:67: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace(race_mapping)
/var/folders/7c/thwjjxtx3_n8jl9xncdtz7rc0000gq/T/ipykernel_77967/330282448.py:65: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice

min_samples 7
min_samples 6
min_samples 4
Detected 3316 rows with missing values. Removed them.
The 'income' column has only two unique values.


/var/folders/7c/thwjjxtx3_n8jl9xncdtz7rc0000gq/T/ipykernel_77967/330282448.py:56: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.df[self.label] = (self.df[self.label].map({self.unfav: 0, self.fav: 1}).astype(int))
/var/folders/7c/thwjjxtx3_n8jl9xncdtz7rc0000gq/T/ipykernel_77967/330282448.py:67: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace(race_mapping)
/var/folders/7c/thwjjxtx3_n8jl9xncdtz7rc0000gq/T/ipykernel_77967/330282448.py:65: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice

min_samples 7
min_samples 6
min_samples 4
Detected 3316 rows with missing values. Removed them.
The 'income' column has only two unique values.


...

min_samples 7
min_samples 6
min_samples 4
Detected 3316 rows with missing values. Removed them.
The 'income' column has only two unique values.


/var/folders/7c/thwjjxtx3_n8jl9xncdtz7rc0000gq/T/ipykernel_77967/330282448.py:56: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.df[self.label] = (self.df[self.label].map({self.unfav: 0, self.fav: 1}).astype(int))
/var/folders/7c/thwjjxtx3_n8jl9xncdtz7rc0000gq/T/ipykernel_77967/330282448.py:67: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace(race_mapping)
/var/folders/7c/thwjjxtx3_n8jl9xncdtz7rc0000gq/T/ipykernel_77967/330282448.py:65: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice

min_samples 7
min_samples 6
min_samples 4
Detected 3316 rows with missing values. Removed them.
The 'income' column has only two unique values.


/var/folders/7c/thwjjxtx3_n8jl9xncdtz7rc0000gq/T/ipykernel_77967/330282448.py:56: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.df[self.label] = (self.df[self.label].map({self.unfav: 0, self.fav: 1}).astype(int))
/var/folders/7c/thwjjxtx3_n8jl9xncdtz7rc0000gq/T/ipykernel_77967/330282448.py:67: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace(race_mapping)
/var/folders/7c/thwjjxtx3_n8jl9xncdtz7rc0000gq/T/ipykernel_77967/330282448.py:65: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice

min_samples 7
min_samples 6
min_samples 4


In [779]:
summary_df

,Dataset,Model,Method,Running Time (s),DI Ratio,AEO Diff,EOD,Consistency,Accuracy,Balanced Accuracy,F1 Score
0,Adult,Logistic Regression,custom_smote_dbscan,13.075679,0.658436,-0.022826,-0.034308,0.933709,0.778377,0.637288,0.444552
